# AIE S2 — Bike Sharing Demand

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s2-bike-demand.ipynb)

**Regression.** Predict how many bikes are rented in a given hour,
from the calendar and the weather.

Challenge: <https://ml-arena.com/viewchallenge/183>

This notebook is the worked example. It runs top to bottom and ends
with a scored submission. Six steps, and every later challenge in
the course is the same six:

1. get the data
2. look at it
3. turn it into numbers — including the holes and the words
4. hold out a validation set **the way you will be scored**
5. fit, measure, predict the test set and submit
6. change the features and do it again

---

## 0. Setup

`pandas`, `seaborn` and `scikit-learn` are already in Colab. The
ML-Arena client is what downloads the data and uploads your answer.

The distribution is called **`mlarena-sdk`** and it imports as
`mlarena`. Do not `pip install mlarena` — that is an unrelated
package by another author, and none of the calls below exist in it.

In [ ]:
!pip install -q mlarena-sdk

---

## 1. Get the data

Paste your personal API key from your ML-Arena **Profile** page. It
starts with `mlk_user_`. `download_dataset` writes the three public
files into the working directory.

In [ ]:
import mlarena

API_KEY = "mlk_user_..."   # <- paste yours here
CHALLENGE_ID = 183

client = mlarena.connect(api_key=API_KEY)
client.download_dataset(CHALLENGE_ID, ".")

---

## 2. Read it

Three files. `X_train` and `y_train` are what you learn from;
`X_test` is what you must predict. `y_test` does not exist on your
side — that is the whole point of the exercise.

One thing to know before you touch anything: **the split is by
time.** The first 80% of the hours are your training set and the
last 20% are held back, so this is a forecast, not a fill-in-the-
blanks. `X_train.csv` therefore ships in calendar order — row 0 is
the earliest hour, the last row is the latest — and you will use
that in step 4.

In [ ]:
import pandas as pd

X_train = pd.read_csv("X_train.csv")
y_train = pd.read_csv("y_train.csv")["prediction"]
X_test = pd.read_csv("X_test.csv")

print("X_train", X_train.shape, " y_train", y_train.shape, " X_test", X_test.shape)
X_train.head()

`X_train` is (13903, 13): **n = 13,903 observations** and 13 columns,
of which one is the `id`, so **p = 12 features**. The target lives in
a separate file, aligned by `id`.

Check the column types before anything else — they decide what you
are allowed to do with each one.

In [ ]:
X_train.dtypes

Four columns are text or boolean (`season`, `holiday`, `workingday`,
`weather`) and the rest are numbers. A linear model cannot consume
the text ones as they are; step 3 is about that.

Now the target and the numeric features:

In [ ]:
print(y_train.describe().round(2))
X_train.describe().round(2)

Look at the `count` row of `describe()` before the statistics. It is
not 13,903 for every column — `describe` drops missing values, so a
short count is your first evidence that some cells are empty. That is
section 3b.

---

## 3. Look at it

This is the step that is easiest to skip and cheapest to do. Six
plots, and each one changes a decision you are about to make.

First, put the target back next to the features so seaborn can plot
them together.

In [ ]:
df = X_train.assign(count=y_train)
df.head(3)

### 3a. The target's distribution

Always look at what you are predicting first.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(9, 3.5))
sns.histplot(df["count"], bins=60, ax=ax, color="#2f6f9f")
ax.set_title(f"Hourly rental count — mean {df['count'].mean():.0f}, median {df['count'].median():.0f}")
ax.set_xlabel("count")
plt.show()

Strongly right-skewed, and bounded below by 0. Two consequences worth
holding on to: the mean sits well above the median, and a model with
an unbounded output — such as a linear one — is free to predict
negative rentals. It will.

### 3b. The holes

Some cells are empty. `sklearn` will not fit through a `NaN`, so this
is not optional cleanup — it is the difference between code that runs
and code that raises.

Count them first, then look at *where* they are.

In [ ]:
X_train.isna().sum()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
sns.heatmap(X_train.isna().T, cbar=False, cmap=["#eef3f8", "#c1553b"],
            ax=ax, xticklabels=False)
ax.set_title("X_train.isna() — one column per hour, in calendar order")
plt.tight_layout()
plt.show()

Two different pictures in one plot, and they mean different things.

- `temp` and `feel_temp` are missing in **solid bands, on the same
  hours** — a thermometer that was down for a couple of days at a
  time. Systematic.
- `windspeed` and `weather` are **scattered** — individual readings
  dropped, closer to at random.

You could only see that because `X_train` is in calendar order. A
shuffled frame would have shown two identical clouds of speckle.

This is also why the two temperature columns must be filled
together: there is no hour where one is available to help the other.

### 3c. The hour of the day

The single most informative plot in this dataset. Split it by
`workingday`, because there is no reason a Tuesday and a Sunday should
look alike.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=df, x="hour", y="count", hue="workingday",
             palette=["#c1553b", "#2f6f9f"], ax=ax)
ax.set_title("Average rentals by hour — working days vs the rest")
ax.set_xticks(range(0, 24, 2))
plt.show()

Two completely different shapes. Working days have a sharp **commute
double peak** at 08:00 and 17:00–18:00; non-working days have a single
broad afternoon hump around 13:00–15:00.

Note what this implies for the model. The relationship between `hour`
and `count` is not a line — it is not even monotone. A model that
multiplies `hour` by one coefficient cannot represent either curve.
Remember this when you read your score; section 6 is where it gets
fixed, and it is the largest single improvement in this notebook.

### 3d. Temperature and weather

The obvious weather candidates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.regplot(data=df, x="temp", y="count", ax=axes[0],
            scatter_kws=dict(alpha=0.08, s=6, color="#2f6f9f"),
            line_kws=dict(color="#c1553b"))
axes[0].set_title("Rentals vs temperature (°C)")
sns.boxplot(data=df, x="weather", y="count", ax=axes[1],
            order=["clear", "misty", "rain", "heavy_rain"], color="#2f6f9f")
axes[1].set_title("Rentals by weather")
plt.tight_layout()
plt.show()

Warmer is busier, roughly linearly — this one a linear model *can*
use. Weather degrades demand in the order you would guess.

`heavy_rain` is worth a second look: the box is almost invisible
because there are only a handful of such hours in the data. A category
with three observations will not support a reliable coefficient — and
note that `value_counts` counts the missing ones too if you ask it to.

In [ ]:
df["weather"].value_counts(dropna=False)

### 3e. What correlates with what

A correlation heatmap on the numeric columns, to catch redundancy
before it reaches the model.

In [ ]:
num = df.select_dtypes("number")
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, annot_kws={"size": 8})
ax.set_title("Correlation, numeric columns")
plt.show()

`temp` and `feel_temp` correlate at **0.99** — they are the same
measurement twice. Keeping both is not fatal here, but it makes the
two coefficients individually meaningless: the fit can trade one
against the other freely.

Now look at the `hour` row: it correlates about **0.4** with `count`,
roughly the same as `temp`. Read that against plot 3c, which showed
`hour` driving the target far harder than any other column. The
correlation understates it badly, because correlation measures only
the *linear* part of a relationship and the hour effect rises, falls,
rises and falls again. A single number cannot see a shape like that —
which is why you plotted it.

### 3f. Season and year

The two columns the *split* makes interesting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.barplot(data=df, x="season", y="count", ax=axes[0],
            order=["spring", "summer", "fall", "winter"], color="#2f6f9f")
axes[0].set_title("Rentals by season")
sns.barplot(data=df, x="year", y="count", ax=axes[1], color="#2f6f9f")
axes[1].set_title("Rentals by year (0 = first, 1 = second)")
plt.tight_layout()
plt.show()

The system grew substantially between the two years. `year` is a
genuine feature here, not noise — and because the test set is the
*end* of year 1, this growth is something your model has to carry
forward rather than average away.

Check what the held-out window actually contains:

In [ ]:
print("train months:", sorted(X_train["month"].unique())[:3], "...",
      "years:", sorted(X_train["year"].unique()))
print("test  months:", sorted(X_test["month"].unique()),
      " years:", sorted(X_test["year"].unique()))
print("test  seasons:", sorted(X_test["season"].dropna().unique()))

No summer, and only the second year. You are being asked to forecast
August to December of year 1 having seen everything before it — which
is exactly what an operator has to do, and is harder than filling in
gaps between hours you already know.

---

## 4. Turn it into numbers

$f(x)$ is a function on real vectors, so two things have to go: the
missing cells and the words.

**Fill with statistics learned on the training rows only.** The median
of `X_train`, applied to both frames. Using `X_test`'s own median
would let information from the test set into your pipeline; your
validation score would improve and your leaderboard score would not.
That gap has a name — leakage.

**The median is the cheap fix, not the best one.** It makes every
cell finite in one line, which is what you need in order to fit
anything at all — but it answers "how warm was it during the outage"
with the median of the whole year. Plot 3b showed those outages are
runs of consecutive hours, and the hour on either side of a run is a
much better guess than an annual median: `ffill()`, or
`interpolate()`, will usually score better here. Start with the
median, get a number on the board, then come back to this line — it
is one of the three ideas in section 9.

`fillna(medians)` names the numeric columns, so it touches only those;
the bare `fillna("unknown")` then catches whatever is left, which by
that point is the text columns.

In [ ]:
medians = X_train.median(numeric_only=True)      # learned on train
print(medians.round(2).to_dict())

print("NaN before:", int(X_train.isna().sum().sum()),
      int(X_test.isna().sum().sum()))
print("NaN after :",
      int(X_train.fillna(medians).fillna("unknown").isna().sum().sum()),
      int(X_test.fillna(medians).fillna("unknown").isna().sum().sum()))

Now the words. `pd.get_dummies` one-hot encodes every non-numeric
column and leaves the numeric ones alone.

The `reindex` on the second line is not optional, and this split makes
it load-bearing: the test window has **no summer hours and no
`heavy_rain`**, so encoding the two frames independently gives them
different columns. `reindex` forces the test matrix onto exactly the
training columns, in the same order.

Both halves go in one function. That is not tidiness — it is what
keeps the rule above true every time you use it, including on the
validation split in section 5, where forgetting it is easy and
invisible.

In [ ]:
def encode(train, test):
    """Impute with the TRAINING frame's medians, one-hot, align columns.

    `train` is whatever you are fitting on — the whole training file in
    section 6, its first 80% in section 5. The medians follow it, so
    the frame being predicted never contributes a statistic to its own
    imputation."""
    medians = train.median(numeric_only=True)
    A = pd.get_dummies(train.fillna(medians).fillna("unknown"))
    B = pd.get_dummies(test.fillna(medians).fillna("unknown"))
    return A, B.reindex(columns=A.columns, fill_value=0)


base_train = X_train.drop(columns=["id"])
base_test = X_test.drop(columns=["id"])
X_train_enc, X_test_enc = encode(base_train, base_test)

print("encoded:", X_train_enc.shape, "->", list(X_train_enc.columns[:6]), "...")
assert list(X_train_enc.columns) == list(X_test_enc.columns)

---

## 5. Hold out a validation set the way you will be scored

You get one score per submission, and you should never learn anything
from the leaderboard you could have learned at home. So hold out part
of the training data and score yourself first.

**How you hold it out matters more than people expect.** The challenge
keeps the last 20% of the hours, so your holdout has to be the last
20% of yours. `X_train` is in calendar order, so that is a slice.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

cut = int(len(base_train) * 0.8)
# encode() again, not a slice of X_train_enc: the medians have to come
# from the first 80% alone, or the held-out fifth helped fill itself.
A_tr, A_va = encode(base_train.iloc[:cut], base_train.iloc[cut:])
b_tr, b_va = y_train.iloc[:cut], y_train.iloc[cut:]

model = LinearRegression().fit(A_tr, b_tr)
pred_va = model.predict(A_va)

print(f"MAE  {mean_absolute_error(b_va, pred_va):.2f}   -> -MAE {-mean_absolute_error(b_va, pred_va):.2f}")
print(f"RMSE {np.sqrt(mean_squared_error(b_va, pred_va)):.2f}")

The challenge ranks on **−MAE** — mean absolute error, negated so that
higher is better — so that is the number to watch. Read it against the
only baseline that matters, predicting the training mean for every
hour:

In [ ]:
baseline = np.full(len(b_va), b_tr.mean())
print(f"predict-the-mean  -MAE {-mean_absolute_error(b_va, baseline):.2f}")
print(f"negative predictions: {(pred_va < 0).sum()} of {len(pred_va)}")

### Why not `train_test_split`?

Because it would lie to you. Try it — same data, same model, the only
difference is that the holdout rows are drawn at random instead of
taken from the end:

In [ ]:
from sklearn.model_selection import train_test_split

R_tr, R_va, c_tr, c_va = train_test_split(
    base_train, y_train, test_size=0.2, random_state=0)
R_tr, R_va = encode(R_tr, R_va)
rand_pred = LinearRegression().fit(R_tr, c_tr).predict(R_va)

print(f"random  holdout  -MAE {-mean_absolute_error(c_va, rand_pred):.2f}")
print(f"time    holdout  -MAE {-mean_absolute_error(b_va, pred_va):.2f}")

About **−97** against about **−146**, and the honest one is the second.
The random holdout keeps 19:00 and 21:00 in training while it asks
about 20:00 — the same weather reading, a count within a few bikes —
so the model interpolates between hours it has already seen instead of
forecasting. That is not the task, and the leaderboard will not agree
with it.

A validation split that does not match the real split is worse than no
validation at all, because you will trust it.

---

## 6. Predict the test set and submit

Refit on **all** the training data now — the holdout has done its job
and more data is better. Then write the two required columns.

In [ ]:
model = LinearRegression().fit(X_train_enc, y_train)   # from section 4
predictions = model.predict(X_test_enc)

submission = pd.DataFrame({"id": X_test["id"], "prediction": predictions})
submission.to_csv("submission.csv", index=False)

print(submission.shape)
submission.head()

Check the file before uploading it. One row per test id, no missing
values, no duplicates:

In [ ]:
assert len(submission) == len(X_test)
assert submission["id"].is_unique
assert submission["prediction"].notna().all()
print("submission.csv looks well-formed")

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"])
print(result)

That scores about **−139**. It is the challenge's declared benchmark:
everything a student who followed this notebook gets, and nothing
more. The rest of the notebook is how you beat it without changing
the model.

---

## 7. Feature engineering

The model is fixed — still `LinearRegression`, still adding its inputs
up. What is not fixed is *what the inputs are*. Two changes, both
measured on the time holdout before they are trusted.

First, section 5 in one function, so that trying an idea is one line
and the two versions differ only in the features.

In [ ]:
def score_time_holdout(train, y):
    """-MAE on the last 20% of the hours — section 5, reusable."""
    cut = int(len(train) * 0.8)
    A, B = encode(train.iloc[:cut], train.iloc[cut:])
    pred = LinearRegression().fit(A, y.iloc[:cut]).predict(B)
    return -mean_absolute_error(y.iloc[cut:], pred)


print(f"baseline                 -MAE {score_time_holdout(base_train, y_train):.2f}")

### 7a. `hour` is not a quantity

This is the one from plot 3c. As it stands, `hour` is a number, so the
model has exactly one coefficient for it and the only statements it
can make are "later is busier" and "later is quieter". The real curve
goes up, down, up and down again.

Make it twenty-four unordered categories instead. `get_dummies` then
gives the model a separate coefficient for every hour of the day.

In [ ]:
fe1_train, fe1_test = base_train.copy(), base_test.copy()
for frame in (fe1_train, fe1_test):
    frame["hour"] = frame["hour"].astype(str)

print(f"hour as 24 categories    -MAE {score_time_holdout(fe1_train, y_train):.2f}")

From about **−146 to −108** on the holdout, with the same model, the
same rows and one `astype(str)`. Nothing in Session 3 — no tree, no
ensemble, no hyperparameter search — buys a jump that size on this
dataset.

The cost is that the model no longer knows 23:00 is next to 00:00. If
that adjacency matters more than the shape does, encode the circle
instead — `np.sin(2*np.pi*hour/24)` and the matching cosine, two
columns rather than twenty-four.

### 7b. Make the missingness a feature, and multiply two columns

Two more hypotheses, written as columns:

- **The hole may be the signal.** You filled `temp` with a median, and
  that threw away the fact that it was ever missing. One binary column
  gives it back and lets the model decide whether it mattered.
- **The effect of one column can depend on another.** Warm weather on
  a working day is a commute; warm weather on a Sunday is an
  afternoon out. A linear model cannot discover that on its own — but
  it can use the product if you hand it one.

The indicators have to be computed **before** the fill, which is why
they are added here and not inside `encode`.

In [ ]:
fe2_train, fe2_test = fe1_train.copy(), fe1_test.copy()
for frame in (fe2_train, fe2_test):
    frame["temp_missing"] = frame["temp"].isna().astype(int)
    frame["wind_missing"] = frame["windspeed"].isna().astype(int)
    frame["weather_missing"] = frame["weather"].isna().astype(int)
    frame["temp_x_working"] = (frame["temp"].fillna(X_train["temp"].median())
                               * frame["workingday"].astype(int))

print(f"+ missing flags + interaction  -MAE {score_time_holdout(fe2_train, y_train):.2f}")

A small gain, and small is the honest result — the flags say the
outages were not informative here, which is worth knowing. Keep them
anyway: they cost four columns and they are the difference between
having tested the idea and having assumed it.

Note what the holdout is being used for. Every number above was
produced without touching the leaderboard, which is the only way to
try five ideas and keep the one that worked.

---

## 8. Submit the engineered model

Refit on all of the training data, predict, and clip: counts cannot be
negative, so a negative prediction is a known error and rounding it up
to zero is free.

In [ ]:
A, B = encode(fe2_train, fe2_test)
predictions = LinearRegression().fit(A, y_train).predict(B)
print(f"negative predictions clipped: {(predictions < 0).sum()}")
predictions = np.clip(predictions, 0, None)

submission = pd.DataFrame({"id": X_test["id"], "prediction": predictions})
submission.to_csv("submission.csv", index=False)
submission.head()

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"])
print(result)

In [ ]:
client.leaderboard(CHALLENGE_ID).head(10)

---

## 9. Your turn

The board should now read about **−99**, against **−139** for the same
model on the raw columns. Everything that closed that gap was a
decision about the features, and none of it changed the model.

The rest of this challenge is yours, and it is not a list to work
through. **Every column you add should come from something you saw in
section 3** — the plots are not decoration, they are where the
hypotheses come from. Go back and read them again with this question:

> Is the shape I can see in this plot something a sum of my columns
> can express? If not, what column would make it one?

Three of the plots answer it out loud:

| What you saw | What it says the model cannot do | The column that fixes it |
|---|---|---|
| 3c — two different daily curves, commute vs weekend | apply one hour effect to both kinds of day | `hour` × `workingday`, or two sets of hour dummies |
| 3a — a long right tail, floored at zero | be wrong symmetrically about a skewed target | fit `np.log1p(y)`, predict `np.expm1` |
| 3b — outages are runs of consecutive hours | know that a filled cell was ever empty | `ffill` instead of the median, and the flags from 7b |

You already have the tools for all three.

### The protocol

This is the part worth keeping for every dataset you ever meet:

1. **One change at a time.** Two at once and you cannot tell which
   one paid.
2. **Score it on the time holdout**, with `score_time_holdout` from
   section 7. Never on the leaderboard — you get one number per
   submission and you learn almost nothing from it.
3. **Keep it only if it moves.** A feature that does nothing is not
   free: it is a coefficient fitted to noise, and it will cost you on
   data you have not seen.
4. **Write down what did not work.** Half of applied ML is knowing
   which ideas are already dead.

The cell below is set up for that loop. Change `engineer`, run it, read
the number, decide.

In [ ]:
def engineer(frame):
    """Your features. Edit this, run the cell, read the number."""
    f = frame.copy()
    f["hour"] = f["hour"].astype(str)
    # --- your ideas below -------------------------------------------
    # f["month"] = f["month"].astype(str)
    # f["is_rush"] = f["hour"].isin(["7", "8", "17", "18"]).astype(int)
    # f = f.drop(columns=["feel_temp"])
    # ----------------------------------------------------------------
    return f


candidate = score_time_holdout(engineer(base_train), y_train)
print(f"your features  -MAE {candidate:.2f}")
print(f"section 7      -MAE {score_time_holdout(fe2_train, y_train):.2f}")
print(f"baseline       -MAE {score_time_holdout(base_train, y_train):.2f}")

When a version beats section 7 on the holdout, re-run section 8 with
`engineer(base_train)` and `engineer(base_test)` in place of
`fe2_train` / `fe2_test`, and submit it.

Two honest warnings. **The holdout is a sample too** — a gain of 0.3
MAE is noise, and chasing it is how you overfit a validation set
instead of a training set. And **the leaderboard is not a search
space**: if you submit forty times and keep the best, you have tuned
on the test set and your number no longer means what it says.

Everything past features — trees, ensembles, hyperparameter search —
is Session 3. It is worth less on this dataset than what you just did,
and that is the lesson.